In [1]:
from IPython.utils.capture import capture_output

with capture_output() as captured:
    %run xraymodel.ipynb

In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import seaborn as sns
from collections import Counter
import pandas as pd

In [3]:
dataset = XRAY_DATASET
images = np.stack([x["img"] for x in XRAY_DATASET])

In [4]:
print(len(dataset))
print(MAX_NUMBER_OF_DATA)

10000
10000


In [5]:
def query_error_with_downprojection(images, error_idx = None, query_idx: int = None, n_features: int = 100, n_neighbors: int = 10, seed=42):

    if query_idx == None:
        if error_idx is None:
            raise ValueError("query_idx or error_idx for FP/TP/FN/TN required!")
        query_idx = np.random.default_rng(seed).choice(error_idx) # random selection

    # Flatten images for pixel-based comparison
    X_raw = images.reshape(len(images), -1).astype(np.float32)

    # PCA
    pca = PCA(n_components=n_features)
    X_pca = pca.fit_transform(X_raw)       

    # UMAP (with PCA features)
    X_umap = umap.UMAP(n_components=2,
                              n_neighbors=n_neighbors,
                              min_dist=0.1,
                              random_state=seed).fit_transform(X_pca)

    # NN
    feature_space = {"Raw": X_raw,
                     "PCA": X_pca,
                     "UMAP": X_umap}

    neighbors = {}

    # TODO: t-SNE OR use downprojector from other notebook

    for space, features in feature_space.items():
        nn = NearestNeighbors(n_neighbors=n_neighbors + 1, metric="euclidean")
        nn.fit(features)
        distances, indices = nn.kneighbors(features[query_idx:query_idx + 1])

        # Exclude query image
        mask = indices[0] != query_idx
        neighbor_indices = indices[0][mask][:n_neighbors]
        neighbor_distances = distances[0][mask][:n_neighbors]
        # Query + nearest neighbors
        plot_indices = [query_idx] + list(neighbor_indices)

        neighbors[space] = neighbor_indices, neighbor_distances, plot_indices

    return query_idx, neighbors, feature_space


In [6]:
def neighbor_overlap(data_1, data_2):
    indices_1, _, _ = data_1
    indices_2, _, _ = data_2

    set_1 = set(np.asarray(indices_1).ravel())
    set_2 = set(np.asarray(indices_2).ravel())

    return len(set_1 & set_2) / len(set_1)

def calculate_overlap_matrix(neighbors: dict):
    methods = list(neighbors)

    return np.array([[neighbor_overlap(neighbors[a], neighbors[b]) for b in methods] for a in methods])

def visualize_overlap(neighbors: dict):
    methods = list(neighbors)

    sns.heatmap(
        calculate_overlap_matrix(neighbors),
        annot=True,
        fmt=".0%",
        xticklabels=methods,
        yticklabels=methods,
        cmap="Blues",
        vmin=0,
        vmax=1,
        square=True)

    plt.title("Shared Neighbors")
    plt.show()

In [7]:
def plot_neighbor_comparison(query_idx, neighbors, images, ground_truth, predictions, threshold=0.5):

    border_colors = {3: "mediumseagreen",
                     2: "orange",
                     1: "lightgray"}

    # Count neighbor overlap across methods (query excluded)
    counts = Counter(
        int(idx)
        for neighbor_indices, _, _ in neighbors.values()
        for idx in set(neighbor_indices)
        if idx != query_idx)

    n_cols = max(len(data[2]) for data in neighbors.values())

    fig, axes = plt.subplots(
        len(neighbors),
        n_cols,
        figsize=(2.4 * n_cols, 3 * len(neighbors)),
        squeeze=False)

    for row, (method, (neighbor_indices, neighbor_distances, plot_indices)) in enumerate(neighbors.items()):

        for col, img_idx in enumerate(plot_indices):

            ax = axes[row, col]
            img_idx = int(img_idx)

            ax.imshow(images[img_idx].squeeze(), cmap="gray")

            classification = classification = ("TP" if ground_truth[img_idx] == 1 and predictions[img_idx] >= threshold else
                                               "FN" if ground_truth[img_idx] == 1 else
                                               "FP" if predictions[img_idx] >= threshold else
                                               "TN")
            gt = ground_truth[img_idx]
            pred = predictions[img_idx]

            if col == 0:
                color = "red"
                title = (
                    f"Query ({classification})\n"
                    f"GT: {gt:.0f} | Pred: {pred:.3f}\n"
                    f"Idx: {img_idx}")
            else:
                color = border_colors[counts[img_idx]]
                title = (
                    f"Neighbor {col} ({classification})\n"
                    f"Dist: {neighbor_distances[col - 1]:,.2f}\n"
                    f"GT: {gt:.0f} | Pred: {pred:.3f}\n"
                    f"Idx: {img_idx}")

            ax.set_title(title, fontsize=9)

            for spine in ax.spines.values():
                spine.set_visible(True)
                spine.set_edgecolor(color)
                spine.set_linewidth(3)

            ax.set_xticks([])
            ax.set_yticks([])

            if col == 0:
                ax.set_ylabel(method, fontsize=12, labelpad=15)

        # Hide unused axes if methods have different neighbor counts
        for col in range(len(plot_indices), n_cols):
            axes[row, col].axis("off")

    #plt.tight_layout()
    #plt.show()

    legend_elements = [Patch(facecolor="red", label="Query"),
                        Patch(facecolor="mediumseagreen", label="Shared by 3 methods"),
                        Patch(facecolor="orange", label="Shared by 2 methods"),
                        Patch(facecolor="lightgray", label="Unique to 1 method")]

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=4,
        frameon=False,
        fontsize=10)

    fig.suptitle("Nearest Neighbor Comparison")

    fig.subplots_adjust(
    top=0.85,      # Space for main title
    bottom=0.10,   # Space for legend
    hspace=0.65,   # Vertical spacing between rows
    wspace=0.12)   # Horizontal spacing between images

    plt.show()

In [8]:
def plot_neighborhood_projections(query_idx, neighbors, feature_space, ground_truth, k=10, show_projected_neighbors=True):
    # Use the first two PCA components
    projections = {
        "PCA": feature_space["PCA"][:, :2],
        "UMAP": feature_space["UMAP"]}

    # Raw-pixel neighbors from your existing calculation
    raw_neighbors = np.asarray(neighbors["Raw"][2])[:k]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, (method, X_2d) in zip(axes, projections.items()):

        # Dataset, colored by ground truth
        scatter = ax.scatter(X_2d[:, 0],
                            X_2d[:, 1],
                            c=ground_truth,
                            cmap="coolwarm",
                            vmin=0,
                            vmax=1,
                            s=12,
                            alpha=0.5)

        # Highlight original raw-pixel neighbors
        ax.scatter(X_2d[raw_neighbors, 0],
                   X_2d[raw_neighbors, 1],
                   s=110,
                   facecolors="none",
                   edgecolors="limegreen",
                   linewidths=2,
                   label="Raw neighbors")

        if show_projected_neighbors:

            nn = NearestNeighbors(n_neighbors=k + 1, 
                                  metric="euclidean").fit(X_2d)

            _, indices = nn.kneighbors(X_2d[query_idx:query_idx + 1])

            projected_neighbors = indices[0][indices[0] != query_idx][:k]

            ax.scatter(X_2d[projected_neighbors, 0],
                       X_2d[projected_neighbors, 1],
                       s=160,
                       facecolors="none",
                       edgecolors="orange",
                       linewidths=1.5,
                       label="2D neighbors")

            overlap = len(set(raw_neighbors) & set(projected_neighbors))

            ax.set_title(f"{method} | Overlap: {overlap}/{k}")

        else:
            ax.set_title(method)

        # Query
        ax.scatter(X_2d[query_idx, 0],
                   X_2d[query_idx, 1],
                   c="red",
                   marker="*",
                   s=250,
                   edgecolors="black",
                   label="Query",
                   zorder=5)

        ax.set_xlabel("Component 1")
        ax.set_ylabel("Component 2")
        ax.legend()

    fig.colorbar(scatter,
                 ax=axes,
                 ticks=[0,1],
                 label="Ground Truth",
                 shrink=0.8)

    plt.show()

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output

pathology_dropdown = widgets.Dropdown(
    options=list(DATASET_LABEL_INDICES_MAP.keys()),
    description="Pathology:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"))

classification_dropdown = widgets.Dropdown(
    options=["FP", "TP", "FN", "TN"],
    value="FP",
    description="Classification:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="250px"))

threshold_slider = widgets.FloatSlider(
    value=0.5,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Threshold:",
    readout_format=".2f",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="350px"))

create_button = widgets.Button(
    description="Create comparison",
    button_style="primary",
    icon="refresh")

output = widgets.Output()

def create_comparison(_):
    with output:
        clear_output(wait=True)

        pathology = pathology_dropdown.value
        classification = classification_dropdown.value
        threshold = threshold_slider.value

        # Dataset and model indices
        dataset_idx = DATASET_LABEL_INDICES_MAP[pathology]
        model_idx = DENSENET121_RES244_CHEX_MODEL.pathologies.index(pathology)

        ground_truth = XRAY_LABELS_LIST[:, dataset_idx]
        predictions = MODEL_SCORE_PREDICTIONS_CONVERTED[:, model_idx]

        # Ignore samples without valid ground truth
        valid = np.isin(ground_truth, [0, 1]) & np.isfinite(predictions)
        positive = predictions >= threshold

        masks = {
            "TP": valid & (ground_truth == 1) & positive,
            "FP": valid & (ground_truth == 0) & positive,
            "TN": valid & (ground_truth == 0) & ~positive,
            "FN": valid & (ground_truth == 1) & ~positive}

        error_idx = np.flatnonzero(masks[classification])

        if len(error_idx) == 0:
            print(f"No {classification} images found for {pathology}.")
            return

        print(f"{pathology} | {classification} with treshold {threshold:.2f}| {len(error_idx)} candidates")

        # Calculate neighbors and projections
        query_idx, neighbors, feature_space = query_error_with_downprojection(
            images=images,
            error_idx=error_idx)

        # Plot 1: Nearest-neighbor image comparison
        plot_neighbor_comparison(
            query_idx=query_idx,
            neighbors=neighbors,
            images=images,
            ground_truth=ground_truth,
            predictions=predictions,
            threshold=threshold)

        # Plot 1.2: Visualize Overlap
        visualize_overlap(neighbors=neighbors)

        # Plot 2: Neighborhood projections
        plot_neighborhood_projections(
            query_idx=query_idx,
            neighbors=neighbors,
            feature_space=feature_space,
            ground_truth=ground_truth,
            k=10)

create_button.on_click(create_comparison)

display(
    widgets.VBox([
        pathology_dropdown,
        classification_dropdown,
        threshold_slider,
        create_button,
        output
    ]))

## Data Augmentation

How does the model respond to data augmentations with regards to

1. Brightness
2. Contrast
3. Gamma
4. Blur
5. Resolution

In [10]:
import torch
import torchvision.transforms.functional as TF

In [11]:
def augment_xray(img, method, strength=0.2):
    x = torch.as_tensor(img, dtype=torch.float32).clone()

    # TODO: Check pixel intensities
    x = ((x + 1024) / 2048).clamp(0, 1)

    match method:
        case  "contrast":
            x = TF.adjust_contrast(x, 1 + strength)
        case "gamma":
            x = TF.adjust_gamma(x, 1 + strength)
        case "blur":
            x = TF.gaussian_blur(x, kernel_size=5, sigma=max(0.1, strength*5))
        case "resolution":
            h, w = x.shape[-2:]
            scale = max(0.1, 1 - strength)
            small = TF.resize(x, [max(1, int(h*scale)), max(1, int(w*scale))], antialias=True)
            x = TF.resize(small, [h,w], antialias=True)

    return (x.clamp(0, 1) * 2048 - 1024).numpy()


In [12]:
def augment_data(augmentations, img, strength=0.2):

    augmented_xrays = {}
    for a in augmentations:
        augmented_xrays[a] = augment_xray(img, a, strength)

    return augmented_xrays


In [13]:
def predict_augmented(augmented_xrays, model, pathology):

    methods = list(augmented_xrays.keys())
    xrays = torch.as_tensor(np.stack([augmented_xrays[m] for m in methods]), dtype=torch.float32)

    device = next(model.parameters()).device
    xrays = xrays.to(device)

    model.eval()
    with torch.inference_mode():
        predictions = model(xrays)

    predictions = predictions.detach().cpu().numpy()

    return dict(zip(methods, predictions[:, pathology]))

In [14]:
def plot_augmentation_comparison(img_idx, img, augmented_xrays, old_prediction, new_predictions, ground_truth, pathology):
    n_cols = len(augmented_xrays) + 1

    fig, axes = plt.subplots(1, n_cols, figsize=(3.5 * n_cols, 4))

    images_to_plot = {"Original": img, **augmented_xrays}

    for ax, (method, image) in zip(axes, images_to_plot.items()):
        ax.imshow(np.asarray(image).squeeze(),
                  cmap="gray",
                  vmin=-1024,
                  vmax=1024)

        new_pred = (
            old_prediction if method == "Original"
            else new_predictions[method])

        ax.set_title(f"{method.capitalize()}\n"
                     f"GT: {ground_truth:.0f}\n"
                     f"Old Pred: {old_prediction:.3f}\n"
                     f"New Pred: {new_pred:.3f}")

        ax.axis("off")

    fig.suptitle(f"Data Augmentation | {pathology} | Image {img_idx}", fontsize=14)

    fig.tight_layout(rect=[0, 0, 1, 0.85])
    plt.show()

In [15]:
augmentation_methods = [None, "contrast", "gamma", "blur", "resolution"]

img_idx_input = widgets.BoundedIntText(value=0,
                                       min=0,
                                       max=len(images) - 1,
                                       description="Image Index:",
                                       style={"description_width": "110px"})

aug_pathology_dropdown = widgets.Dropdown(options=list(DATASET_LABEL_INDICES_MAP.keys()),
                                          description="Pathology:",
                                          style={"description_width": "110px"},
                                          layout=widgets.Layout(width="350px"))

strength_slider = widgets.FloatSlider(value=0.2,
                                      min=0.05,
                                      max=0.8,
                                      step=0.05,
                                      description="Strength:",
                                      style={"description_width": "110px"},
                                      layout=widgets.Layout(width="350px"))

augment_button = widgets.Button(description="Create augmentations",
                                button_style="primary",
                                icon="refresh")

augmentation_output = widgets.Output()


def create_augmentations(_):
    with augmentation_output:
        clear_output(wait=True)

        img_idx = img_idx_input.value
        pathology = aug_pathology_dropdown.value
        strength = strength_slider.value

        dataset_idx = DATASET_LABEL_INDICES_MAP[pathology]
        model_idx = DENSENET121_RES244_CHEX_MODEL.pathologies.index(pathology)

        img = images[img_idx]
        ground_truth = XRAY_LABELS_LIST[img_idx, dataset_idx]
        old_prediction = MODEL_SCORE_PREDICTIONS_CONVERTED[img_idx, model_idx]

        augmented_xrays = augment_data(augmentation_methods, img, strength=strength)

        new_predictions = predict_augmented(augmented_xrays, DENSENET121_RES244_CHEX_MODEL, model_idx)

        plot_augmentation_comparison(img_idx=img_idx,
                                     img=img,
                                     augmented_xrays=augmented_xrays,
                                     old_prediction=old_prediction,
                                     new_predictions=new_predictions,
                                     ground_truth=ground_truth,
                                     pathology=pathology)


augment_button.on_click(create_augmentations)

display(widgets.VBox([img_idx_input,
                      aug_pathology_dropdown,
                      strength_slider,
                      augment_button,
                      augmentation_output]))

In [16]:
class AugmentedXRayDataset(Dataset):
    def __init__(self, images, method=None, strength=0.2):
        self.images = images
        self.method = method
        self.strength = strength

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]

        if self.method is not None:
            img = augment_xray(img, self.method, self.strength)
    
        return torch.as_tensor(img, dtype=torch.float32)

In [17]:
datasets = {method or "Original": AugmentedXRayDataset(images, method=method, strength=0.2) for method in augmentation_methods}

In [18]:
def process_augmented_dataset(dataset, model, batch_size=32):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    n = len(dataset)
    offset = 0

    all_predictions = []
    features = np.empty((n, *dataset[0].shape), dtype=np.float32)

    for batch in loader:
        y = model(batch)

        b = len(batch)

        features[offset:offset + b] = batch.cpu().numpy()
        all_predictions.append(y.detach().cpu())

        offset += b

    return features, np.concatenate(all_predictions, axis=0)

In [19]:
def compare_augmentations(datsets, 
                          projection_method, 
                          visualization_fns, 
                          visualization_names,
                          ground_truth, 
                          dataset_label_indices_map,
                          model=DENSENET121_RES244_CHEX_MODEL,
                          dataset_gender_list=XRAY_GENDER_LIST,
                          dataset_ages=XRAY_AGE_LIST,
                          pathologies=DATASET_PATHOLOGIES,
                          threshold=0.7):

    results = {}

    for method, dataset in datasets.items():
        print(f"Processing: {method}")

        features, predictions = process_augmented_dataset(dataset, model)

        downprojector = Downprojector(features.reshape(-1, 224* 224))

        #fig = create_multi_pathology_viewer(downprojector=downprojector,
        #                                    projection_method=projection_method,
        #                                    visualization_fns=visualization_fns,
        #                                    visualization_names=visualization_names,
        #                                    model_score_predictions=predictions,
        #                                    ground_truth=ground_truth,
        #                                    dataset_label_indices_map=dataset_label_indices_map,
        #                                    dataset_gender_list=dataset_gender_list,
        #                                    dataset_ages=dataset_ages,
        #                                    threshold=threshold)

        results[method] = {"features": features, "predictions": predictions} #"figure":fig}

    return results

In [ ]:
print(len(dataset))

In [ ]:
comparison = compare_augmentations(datasets, "S-UMAP", visualization_functions, visualization_names, XRAY_LABELS_LIST, DATASET_LABEL_INDICES_MAP)

Processing: Original


In [ ]:
def compare_prediction_scores(results, pathologies, model_pathologies):
    original = results["Original"]["predictions"]
    rows = []

    for method, result in results.items():
        if method == "Original":
            continue

        predictions = result["predictions"]

        for pathology in pathologies:
            model_idx = model_pathologies.index(pathology)

            delta = predictions[:, model_idx] - original[:, model_idx]

            rows.append({
                "Pathology": pathology,
                "Augmentation": method,
                "Mean Original": np.nanmean(original[:, model_idx]),
                "Mean Augmented": np.nanmean(predictions[:, model_idx]),
                "Mean Delta": np.nanmean(delta),
                "Mean Absolute Delta": np.nanmean(np.abs(delta)),
                "Max Absolute Delta": np.nanmax(np.abs(delta))})

    return pd.DataFrame(rows)

In [ ]:
score_comparison = compare_prediction_scores(comparison, DATASET_PATHOLOGIES, MODEL_PATHOLOGIES)

display(score_comparison)

In [ ]:
pivot = score_comparison.pivot(index="Pathology",
                               columns="Augmentation",
                               values="Mean Absolute Delta")

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(pivot.values, cmap="viridis", aspect="auto")

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns)

ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)

fig.colorbar(im, ax=ax, label="Mean Absolute Prediction Change")

ax.set_title("Prediction Sensitivity Across Pathologies")
plt.tight_layout()
plt.show()